In [1]:

!pip install transformers torch

In [2]:
pip install datasets

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch
import math
from tqdm import tqdm

In [4]:
# 1. Load a text dataset (we use the IMDB dataset)
dataset = load_dataset("imdb", split="train")

# Let's also check the features to find the text column name
print(dataset.features)

print(f"Number of examples in dataset: {len(dataset)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}
Number of examples in dataset: 25000


In [5]:
# 2. Initialize a tokenizer (we'll use GPT-2's tokenizer for compatibility with a GPT-2 model)
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 doesn't have a pad by default

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [6]:
# 3. Tokenize the dataset efficiently using `.map` with batched processing
def tokenize_function(examples):
    return tokenizer(examples["text"], return_special_tokens_mask=False)

tokenized_ds = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
# The dataset now has columns like 'input_ids' and 'attention_mask'

print(tokenized_ds[0]["input_ids"][:20])  # print first 20 token IDs of first example for sanity check

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 1024). Running this sequence through the model will result in indexing errors


[40, 26399, 314, 3001, 327, 47269, 20958, 12, 56, 23304, 3913, 422, 616, 2008, 3650, 780, 286, 477, 262, 10386]


In [7]:
# 3. Tokenize the dataset by padding/truncating each example
block_size = 256 # We can define this here now

def tokenize_function(examples):
    # This will pad or truncate every individual text to block_size
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=block_size,
        return_special_tokens_mask=False # We'll get attention_mask instead
    )

tokenized_ds = dataset.map(tokenize_function, batched=True, remove_columns=["text", "label"])

# Let's check the new columns
print(tokenized_ds)
# Should show 'input_ids' and 'attention_mask'

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 25000
})


In [8]:
# 5. Create a DataLoader (was Cell 7)
def collate_fn(batch):
    input_ids = torch.tensor([example["input_ids"] for example in batch], dtype=torch.long)
    attention_mask = torch.tensor([example["attention_mask"] for example in batch], dtype=torch.long)

    return {
        "input_ids": input_ids,
        "labels": input_ids.clone(), # Labels are still the input_ids
        "attention_mask": attention_mask
    }

# We use tokenized_ds now, not lm_ds
train_loader = DataLoader(tokenized_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)

In [9]:
# 6. Iterate through a couple of batches (was Cell 8)
for batch in train_loader:
    print("Input IDs shape:", batch["input_ids"].shape)
    print("Labels shape:", batch["labels"].shape)
    print("Attention Mask shape:", batch["attention_mask"].shape)
    break

Input IDs shape: torch.Size([16, 256])
Labels shape: torch.Size([16, 256])
Attention Mask shape: torch.Size([16, 256])


In [10]:
from transformers import AutoModelForCausalLM
from torch.optim import AdamW

In [11]:
# 7. Load Model and Optimizer
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
print(f"Using device: {device}")

# Load the model configured for Causal Language Modeling
model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)

# Set up the optimizer
optimizer = AdamW(model.parameters(), lr=5e-5) # 5e-5 is a common learning rate

# 8. Run a Single Training Step
print("Running a single training step...")

# Set model to training mode
model.train()

# Get one batch from the dataloader
try:
    batch = next(iter(train_loader))
except Exception as e:
    print(f"Error getting batch: {e}")
    print("Make sure you have run all previous cells, especially the DataLoader one.")

# Move the batch to the same device as the model
batch = {k: v.to(device) for k, v in batch.items()}

# Clear old gradients
optimizer.zero_grad()

# --- Forward Pass ---
# Feed the batch into the model
# The model automatically uses the 'labels' to calculate the loss
# The attention_mask ensures it ignores padding
outputs = model(**batch)

# Get the loss
loss = outputs.loss

# --- Backward Pass ---
# Calculate the gradients
loss.backward()

# --- Optimizer Step ---
# Update the model's weights
optimizer.step()

print(f"Single batch complete.")
print(f"Loss: {loss.item()}")

Using device: cuda


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Running a single training step...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Single batch complete.
Loss: 5.914530277252197


In [12]:
# 9. Load and Prepare Test Data
print("Loading test dataset...")
# Load the 'test' split
test_dataset = load_dataset("imdb", split="test")

# Tokenize the test data using the *same* function
# (This assumes tokenize_function is still in memory from Cell 5)
tokenized_test_ds = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text", "label"]
)

# Create a test DataLoader using the *same* collate_fn
# (This assumes collate_fn is still in memory from Cell 7)
test_loader = DataLoader(
    tokenized_test_ds,
    batch_size=16, # Can use the same batch size
    shuffle=False, # No need to shuffle for evaluation
    collate_fn=collate_fn
)

print(f"Test data loaded. Number of batches: {len(test_loader)}")

Loading test dataset...


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Test data loaded. Number of batches: 1563


Evaluating perplexity...


100%|██████████| 1563/1563 [05:43<00:00,  4.55it/s]

------------------------------
Evaluation Complete.
Average Loss: 5.0423
Perplexity: 154.8215


In [17]:
# 11. Run One Full Training Epoch
print("--- Starting Training Epoch 1 ---")
# (Make sure tqdm is imported: from tqdm import tqdm)

model.train()  # Set model to training mode

total_train_loss = 0

# Wrap train_loader with tqdm for a progress bar
for batch in tqdm(train_loader):

    # Move batch to device
    batch = {k: v.to(device) for k, v in batch.items()}

    # Clear old gradients
    optimizer.zero_grad()

    # --- Forward Pass ---
    outputs = model(**batch)
    loss = outputs.loss

    # --- Backward Pass ---
    loss.backward()

    # --- Optimizer Step ---
    optimizer.step()

    total_train_loss += loss.item()

avg_train_loss = total_train_loss / len(train_loader)
print(f"--- Epoch 1 Complete ---")
print(f"Average Training Loss: {avg_train_loss:.4f}")

--- Starting Training Epoch 1 ---


100%|██████████| 1563/1563 [19:03<00:00,  1.37it/s]

--- Epoch 1 Complete ---
Average Training Loss: 3.0952


In [18]:
# 10. Calculate Perplexity
print("Evaluating perplexity...")

model.eval()

total_loss = 0
total_batches = 0

# --- IMPORTANT: Disable gradient calculations ---
# We don't need to calculate gradients, which saves memory and compute.
with torch.no_grad():

    # Loop over all batches in the test_loader
    for batch in tqdm(test_loader):

        # Move batch to the device
        batch = {k: v.to(device) for k, v in batch.items()}

        # Perform a forward pass
        outputs = model(**batch)

        # Get the loss
        loss = outputs.loss

        # Add the loss for this batch to our total
        total_loss += loss.item()
        total_batches += 1

# Calculate the average loss over all batches
avg_loss = total_loss / total_batches

# Calculate perplexity
perplexity = math.exp(avg_loss)

print("---" * 10)
print(f"Evaluation Complete.")
print(f"Average Loss: {avg_loss:.4f}")
print(f"Perplexity: {perplexity:.4f}")

Evaluating perplexity...


100%|██████████| 1563/1563 [05:48<00:00,  4.49it/s]

------------------------------
Evaluation Complete.
Average Loss: 2.9539
Perplexity: 19.1798
